# Multimodal Music Analysis

The target of this project is the implementation of a complete pipeline for extracting knowledge from music data by combining two different modalities: **Audio** and **Text**. The project is split into two parts:  
1. Preprocessing of the data, generation of embeddings and researching analysis.
2. Classification, Fusion and Evaluation.  
  
This notebook contains the implementation of part 1.


Requirements:

In [2]:
%pip install seaborn
%pip install datasets
%pip install matplotlib
%pip install vaderSentiment
%pip install kneed
%pip install contractions
%pip install pyspellchecker
%pip install scikit-learn
%pip install pandas

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.1 MB 6.7 MB/s eta 0:00:02
   ----------- ---------------------------- 2.4/8.1 MB 7.5 MB/s eta 0:00:01
   -------------------- ------------------- 4.2/8.1 MB 8.1 MB/s eta 0:00:01
   ---------------------------- ----------- 5.8/8.1 MB 7.8 MB/s eta 0:00:01
   -------------------------------------- - 7.9/8.1 MB 8.2 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 7.7 MB/s  0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ------------------------------- -------- 1.8/2.3 MB 11.2 MB/s eta 0:00:01
   ------------------

## 1. Data Collection


In [45]:
import pandas as pd
import os

# --- Importing the datasets ---
genres_df = pd.read_csv('data/id_genres.csv', sep='\t') 
information_df = pd.read_csv('data/id_information.csv', sep='\t')
tags_df = pd.read_csv('data/id_tags.csv', sep='\t')
mfcc_stats_df = pd.read_csv('data/id_mfcc_stats.tsv', sep='\t').dropna()


# --- Filtering the dataset ---

# filtering the genres dataset to only include the top 5 most common genres
top_genres = genres_df['genres'].value_counts().head(5).index.to_list()
filtered_genres_df = genres_df[genres_df['genres'].isin(top_genres)]

# --- Filtering the lyrics of the songs ---

lyrics_folder = 'data/processed_lyrics'

target_ids = set(filtered_genres_df['id'].astype(str))

lyrics_data = []

for filename in os.listdir(lyrics_folder):
    if filename.endswith('.txt'):
        song_id = filename.replace('.txt', '')
        
        if song_id in target_ids:
            file_path = os.path.join(lyrics_folder, filename)
            
            with open(file_path, 'r', encoding='utf-8') as f:
                lyrics_text = f.read()
            
            lyrics_data.append({
                'id': song_id,
                'lyrics': lyrics_text
            })

lyrics_df = pd.DataFrame(lyrics_data)

# --- Merging the datasets ---

df = pd.merge(filtered_genres_df, information_df, on='id')
df = pd.merge(df, tags_df, on='id')
df = pd.merge(df, mfcc_stats_df, on='id')
df = pd.merge(df, lyrics_df, on='id')

# --- Cleaning the dataset ---
df = df[df['lyrics'].notna()]
df = df[df['lyrics'].str.strip() != '']
df = df[df['genres'].notna()]
df = df[df['genres'].str.strip() != '']


df['id'].count()





np.int64(11571)